In [25]:
import pandas as pd

## 1. Loading Data and initialization of *Silver Layer*

In [26]:
#path to the bronze reviews CSV file
file_path = r'C:\Users\Maciek\Desktop\netflixdb\databases\reviews.csv'

#Loading raw data from the bronze layer
df_bronze = pd.read_csv(r'C:\Users\Maciek\Desktop\netflixdb\databases\reviews.csv')

#Verification
print(f"Number of columns in df: {df_bronze.shape[1]}")
print("First 5 rows: ")
print(df_bronze.head())

#Creating a copy of the bronze dataframe to work on
df_silver = df_bronze.copy()

Number of columns in df: 12
First 5 rows: 
       review_id     user_id    movie_id  rating review_date device_type  \
0  review_000001  user_07066  movie_0360       4  2025-03-29      Mobile   
1  review_000002  user_02953  movie_0095       5  2024-07-19      Mobile   
2  review_000003  user_05528  movie_0518       4  2025-02-11      Tablet   
3  review_000004  user_07612  movie_0672       5  2025-11-26      Mobile   
4  review_000005  user_03424  movie_0580       3  2025-07-11      Mobile   

   is_verified_watch  helpful_votes  total_votes  \
0              False            3.0          5.0   
1               True            2.0          2.0   
2               True            2.0          5.0   
3               True            7.0          7.0   
4               True            1.0          5.0   

                                         review_text sentiment  \
0          Fantastic cinematography and plot twists.  positive   
1                      This series is a masterpiece!  p

## 2. Analyses and Exploration Nulls

In [27]:
df_silver.head()
df_silver.isnull().sum()

review_id               0
user_id                 0
movie_id                0
rating                  0
review_date             0
device_type             0
is_verified_watch       0
helpful_votes        1815
total_votes          1815
review_text           785
sentiment               0
sentiment_score      1209
dtype: int64

## 3. Analyses of helpful_votes and total_votes columns and covariance

In [28]:
#correlation between helpful_votes and total_votes NULLs
print(df_silver[['helpful_votes','total_votes']].isnull().sum())

# Unique values in helpful_votes and total_votes
print(df_silver['helpful_votes'].unique())
print(df_silver['total_votes'].unique())

helpful_votes    1815
total_votes      1815
dtype: int64
[ 3.  2.  7.  1.  4.  5. nan  6.  0.  8.  9. 10. 13. 12. 11.]
[ 5.  2.  7.  4. 10.  9. nan  6.  3.  8. 11. 13.  0. 12.  1. 15. 14. 16.]


## 4. Cleaning and Imputation of empirical data in 'helpful_votes' column

In [29]:
# Feature Engineering: Create a binary flag (1/0) indicating where the original value was missing.
# This is crucial for ML models to capture the predictive power of missing data itself.
df_silver['is_helpful_votes_missing'] = df_silver['helpful_votes'].isnull().astype(int)

# Imputation Strategy: Fill missing helpful_votes with 0, assuming no helpful votes were recorded.
df_silver['helpful_votes'] = df_silver['helpful_votes'].fillna(0)

#Verification
df_silver['helpful_votes'].isnull().sum()

np.int64(0)

## 5. Cleaning and Imputation of empirical data in 'total_votes' column

In [30]:
#Create a binary flag (1/0) indicating where the original value was missing.
df_silver['is_total_votes_missing'] = df_silver['total_votes'].isnull().astype(int)

# Imputation Strategy: Fill missing total_votes with 0, assuming no total votes were recorded.
df_silver['total_votes'] = df_silver['total_votes'].fillna(0)

#Verification
df_silver['total_votes'].isnull().sum()

np.int64(0)

## 6. Cleaning and Imputation of string data

In [31]:
#Filling missing review_text with 'NO_REVIEW' placeholder
df_silver['review_text'] = df_silver['review_text'].fillna('NO_REVIEW')

#Verification
print(df_silver['review_text'].isnull().sum())

0


## 7. Cleaning and Imputation of Categorical data

In [32]:
#Group by sentiment to see the range of sentiment_score values
print(df_silver.groupby('sentiment')['sentiment_score'].agg(['min','max']))

#Calculate medians for each sentiment category
negative_median = df_silver.loc[df_silver['sentiment'] == 'negative', 'sentiment_score'].median()
neutral_median = df_silver.loc[df_silver['sentiment'] == 'neutral', 'sentiment_score'].median()
positive_median = df_silver.loc[df_silver['sentiment'] == 'positive', 'sentiment_score'].median()

print("medians:", dict(negative=negative_median, neutral=neutral_median, positive=positive_median))

#Impute missing sentiment_score with median based on sentiment category
df_silver.loc[(df_silver['sentiment'] == 'negative') & (df_silver['sentiment_score'].isna()), 'sentiment_score'] = negative_median
df_silver.loc[(df_silver['sentiment'] == 'neutral') & (df_silver['sentiment_score'].isna()), 'sentiment_score'] = neutral_median
df_silver.loc[(df_silver['sentiment'] == 'positive') & (df_silver['sentiment_score'].isna()), 'sentiment_score'] = positive_median

#Verification
df_silver['sentiment_score'].isnull().sum()


           min  max
sentiment          
negative   0.0  0.4
neutral    0.4  0.6
positive   0.6  1.0
medians: {'negative': np.float64(0.195), 'neutral': np.float64(0.503), 'positive': np.float64(0.8005)}


np.int64(0)

## 7. Final Output: Saving Silver Layer to CSV

In [33]:
output_file = 'netflix_silver_layer_reviews.csv'

df_silver.to_csv(
    output_file,
    sep = ',',
    index = False
)